# 02. Embeddings & Vector Store
---

# Información

## Objetivo

Generar embeddings y construir un vector store utilizando información oficial del Plan Comunal de Emergencia de Valdivia.

## Objetivos específicos

- Cargar el documento limpio generado en la etapa de ingestión.
- Crear fragmentos de texto para búsqueda semántica.
- Transformar los fragmentos en embeddings.
- Construir un índice vectorial.

## Entradas

- data/processed/Plan-Comunal-de-Emergencia-2025-2.json

## Salidas

- Embeddings generados.
- Vector store para recuperación semántica.



## Etapa 1: Generación de Embeddings

En esta etapa se transformará el contenido textual de cada `Chunk` en una representación vectorial utilizando un modelo de Sentence Transformers.

Los embeddings permiten representar el significado semántico del texto mediante vectores numéricos. Estas representaciones constituyen la base de la búsqueda semántica utilizada por el sistema RAG.

### 1. Definir ubicación del archivo procesado

In [1]:
from pathlib import Path
import sys

# Agregar la raíz del proyecto al PATH para importar módulos de src
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
vector_store_path = project_root / "data" / "vector_store"
print(vector_store_path)

c:\ALURA_AI_Proyect\Challege_Alura_ONE_AI_FOR_TECH\Challenge_Alura_Agente\data\vector_store


### 2. Carga del documento

En esta sección se carga un documento PDF de prueba utilizando el módulo `load_pdf`, desarrollado durante la etapa de Ingesta. El objetivo es obtener un objeto `Document` que servirá como punto de partida para el resto del pipeline.

Para esta validación se utilizará el documento **test_plan_comunal.pdf**, el mismo empleado en las pruebas de las etapas anteriores, garantizando la consistencia durante el desarrollo del proyecto.

In [3]:
import json

processed_file = Path(
    "../data/processed/Plan-Comunal-de-Emergencia-2025-2.json"
)

with processed_file.open(
    "r",
    encoding="utf-8"
) as file:
    document_data = json.load(file)

document_data.keys()

dict_keys(['id', 'title', 'source', 'file_type', 'content', 'metadata'])

Validación de la carga del documento:

In [4]:
from src.models.document import Document

document = Document(
    id=document_data["id"],
    title=document_data["title"],
    source=document_data["source"],
    file_type=document_data["file_type"],
    content=document_data["content"],
    metadata=document_data["metadata"]
)

print(f"Título                 : {document.title}")
print(f"Fuente                 : {document.source}")
print(f"Tipo de archivo        : {document.file_type}")
print(f"Longitud del contenido : {len(document.content)} caracteres")
print(f"Cantidad de metadatos  : {len(document.metadata)}")

Título                 : Plan-Comunal-de-Emergencia-2025-2
Fuente                 : ..\data\raw\muni_valdivia\planes\Plan-Comunal-de-Emergencia-2025-2.pdf
Tipo de archivo        : pdf
Longitud del contenido : 90189 caracteres
Cantidad de metadatos  : 1


---
## Etapa 2. Preparación de fragmentos de texto (Chunks)

### 1. Creación de fragmentos de texto

In [5]:
import src.processing.chunker as chunker
chunks = chunker.create_chunks(document)
len(chunks)

c:\ALURA_AI_Proyect\Challege_Alura_ONE_AI_FOR_TECH\Challenge_Alura_Agente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


229

### 2. Validación de fragmentos

In [6]:
print(f"Cantidad de chunks: {len(chunks)}")

first_chunk = chunks[0]

print(f"ID: {first_chunk.id}")
print(f"Documento origen: {first_chunk.document_id}")
print(f"Índice: {first_chunk.chunk_index}")
print(f"Longitud: {len(first_chunk.content)} caracteres")

print("\nPrimeros 150 caracteres:\n")
print(first_chunk.content[:150])

Cantidad de chunks: 229
ID: ad49643a-fe7f-4751-b9c8-94e9921791f6
Documento origen: 223fa9f7-3c40-4c47-9232-e64f8ece8b04
Índice: 0
Longitud: 499 caracteres

Primeros 150 caracteres:

Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página 
1 de 33 
Fecha: Agosto-2025 


---
## Etapa 3. Generación de embeddings

### 1. Carga de modelo de embeddings

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4469.74it/s]


### 2. Generación de embeddings para cada fragmento

In [8]:
import src.processing.embedder as embedder

embedded_chunks = embedder.generate_embeddings(
    chunks=chunks,
    model=model,
)

print(f"Chunks con embeddings: {len(embedded_chunks)}")

Chunks con embeddings: 229


### 3. Validar el primer embedding generado

In [9]:
first_chunk = embedded_chunks[0]

print(type(first_chunk.embedding))
print(len(first_chunk.embedding))

<class 'numpy.ndarray'>
384


### 4. Validación adicional

In [10]:
all_embedding = all(
    chunk.embedding is not None
    for chunk in embedded_chunks
)

print(f"Todos los chunks tienen embedding: {all_embedding}")

Todos los chunks tienen embedding: True


In [11]:
first_embedding = embedded_chunks[0].embedding

print(type(first_embedding))
print(first_embedding.shape)
print(first_embedding[:10])

<class 'numpy.ndarray'>
(384,)
[ 0.08619918  0.00378665  0.01216879  0.02605192 -0.01683555 -0.05457666
 -0.1163444   0.0337018  -0.03534188  0.03167299]


---
## Etapa 4. Construcción del vector store

ETAPA 1 - Ingestion
        ✅ Document

ETAPA 2 - Chunking
        ✅ 229 Chunk

ETAPA 3 - Embeddings
        ✅ 229 vectores semánticos

ETAPA 4 - Vector Store (CHROMADB)
        🚧 En desarrollo

Objetivo de esta etapa

Construir un índice vectorial que permita almacenar los embeddings generados y realizar búsquedas por similitud semántica.

### 1. Persistencia del vector store

In [12]:
import src.processing.vector_store as vector_store
from src.processing.vector_store import get_or_create_collection

### 2. Crear la colección

In [13]:
collection = get_or_create_collection(
    collection_name="alessia_collection",
    vector_store_path=vector_store_path
)

print(f"Nombre de la colección: {collection.name}")
print(f"Cantidad de vectores en la colección: {collection.count()}")

Nombre de la colección: alessia_collection
Cantidad de vectores en la colección: 229


### 3. Indexar los fragmentos con embeddings

**Decisión de diseño:** Durante la versión 1.0 de SOPHIA, la indexación del Vector Store se ejecuta únicamente cuando la colección no contiene datos. Esta estrategia evita la duplicación de registros durante el desarrollo y mantiene el notebook reproducible. En versiones futuras, este mecanismo será reemplazado por una estrategia de actualización del índice basada en la detección de cambios en los documentos de origen.

In [14]:
if collection.count() == 0:
    vector_store.index_chunks(
        chunks=embedded_chunks,
        collection=collection,
    )

    print(f"Vectores indexados: {collection.count()}")

else:
    print(
        f"La colección '{collection.name}' ya contiene "
        f"{collection.count()} vectores. "
        "Se reutilizará el índice existente."
    )

La colección 'alessia_collection' ya contiene 229 vectores. Se reutilizará el índice existente.


## ✅ Conclusiones

En esta etapa aprendimos que:

- El documento limpio generado durante la etapa de Ingestión puede reutilizarse sin repetir el procesamiento del PDF.
- El proceso de chunking divide el documento en fragmentos manteniendo la trazabilidad mediante identificadores y metadatos.
- Cada fragmento puede transformarse en una representación vectorial utilizando Sentence Transformers.
- Los embeddings permiten construir un índice vectorial persistente con ChromaDB, que será utilizado en la siguiente etapa para realizar búsquedas semánticas.
- La persistencia del Vector Store permite reutilizar los embeddings sin necesidad de regenerarlos en cada ejecución.

In [15]:
print(collection.name)
print(collection.count())

data = collection.get()
print(len(data["ids"]))
print(data["ids"][:3])

alessia_collection
229
229
['9fe9ed3a-a701-41cb-8354-077908d00302', 'ecb58812-e381-43d3-a1bb-7f54707359e4', 'f17c1d0b-a19a-4e24-834e-14acb21f2886']
